In [37]:
from torch.utils.data import DataLoader, random_split 
from firealarm_net import FireAlarmCNN, MelDataset
from sklearn.preprocessing import LabelEncoder 
import torch.nn as nn
import numpy as np 
import torch 
import os

In [38]:
# Set seed to have the same outputs constantly when testing
seed = 42
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
np.random.seed(seed)

In [ ]:
# Setting up our features
# Load Data by traversing through the features
features = []
labels = []

print("Loading data...")
for filename in sorted(os.listdir("features")):
    if filename.endswith(".npy"):
        # Simple label extraction based on filename
        if "siren" in filename: 
            label = "siren"
        elif "appliance" in filename: 
            label = "appliance"
        elif "carbon" in filename: 
            label = "carbon"
        elif "smoke" in filename: 
            label = "smoke"
        else: 
            continue
        
        data = np.load(os.path.join("features", filename))
        features.append(data)
        labels.append(label)

        if label == "carbon":
            for _ in range(6):  
                features.append(data)
                labels.append(label)

# Convert to Tensors
features = np.array(features)

# Standardize data (make mean 0, std 1) for better learning
#features = (features - features.mean()) / (features.std() + 1e-8)
mean = features.mean(axis=(1, 2), keepdims=True)
std = features.std(axis=(1, 2), keepdims=True)
features = (features - mean) / (std + 1e-8)

# Add channel dim: (N, 64, 32) -> (N, 1, 64, 32)
features = features[:, None, :, :]

# Encode Labels (appliance -> 0, carbon -> 1, siren -> 2, smoke -> 3)
encoder = LabelEncoder()
labels_encoded = encoder.fit_transform(labels)

print(f"Data loaded: {len(features)} samples.")

Loading data...
Data loaded: 2410 samples.


In [40]:
# Preparing for our Model
# Dataset & Loader
dataset = MelDataset(features, labels_encoded)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size], generator=torch.Generator().manual_seed(seed))

# Create a generator for DataLoader shuffling
g = torch.Generator()
g.manual_seed(seed)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# Model setup
model = FireAlarmCNN(num_classes=len(encoder.classes_))
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss() # Standard loss, no weights

In [41]:
# Training the Model with Model Checkpointing
epochs = 30
best_acc = 0.0
patience_counter = 0
patience = 5  # Stop if no improvement for 5 epochs

print("\nStarting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        output = model(x)
        loss = criterion(output, y)
        loss.backward()
        optimizer.step()
        
        # Track training stats
        running_loss += loss.item() * x.size(0)
        correct += (output.argmax(1) == y).sum().item()
        total += y.size(0)
    
    train_loss = running_loss / total
    train_acc = correct / total
    
    # Validation step
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (out.argmax(1) == y).sum().item()
            val_total += y.size(0)
            
    val_loss = val_loss / val_total
    val_acc = val_correct / val_total
    
    print(f"Epoch {epoch+1}/{epochs}  "
          f"TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}")
    
    # Model Checkpointing: Save only on validation accuracy improvement
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'model/model.pt')
        print(f"Checkpoint saved! New best accuracy: {best_acc:.4f}")

print(f"\nTraining Complete!")
print(f"Best Accuracy: {best_acc:.4f}")
print(f"Model weights saved to 'model/model.pt'")


# Model Stats: Epoch 29/30  TrainLoss=0.2685  ValLoss=0.1966  TrainAcc=0.9039  ValAcc=0.9357


Starting training...
Epoch 1/30  TrainLoss=0.9234  ValLoss=0.7112  TrainAcc=0.6266  ValAcc=0.7448
Checkpoint saved! New best accuracy: 0.7448
Epoch 2/30  TrainLoss=0.5895  ValLoss=0.5288  TrainAcc=0.7962  ValAcc=0.8734
Checkpoint saved! New best accuracy: 0.8734
Epoch 3/30  TrainLoss=0.4341  ValLoss=0.4081  TrainAcc=0.8667  ValAcc=0.8548
Epoch 4/30  TrainLoss=0.3397  ValLoss=0.2986  TrainAcc=0.9061  ValAcc=0.9149
Checkpoint saved! New best accuracy: 0.9149
Epoch 5/30  TrainLoss=0.2896  ValLoss=0.3963  TrainAcc=0.9149  ValAcc=0.8527
Epoch 6/30  TrainLoss=0.2464  ValLoss=0.2591  TrainAcc=0.9305  ValAcc=0.8963
Epoch 7/30  TrainLoss=0.2174  ValLoss=0.2461  TrainAcc=0.9409  ValAcc=0.9170
Checkpoint saved! New best accuracy: 0.9170
Epoch 8/30  TrainLoss=0.1945  ValLoss=0.2861  TrainAcc=0.9466  ValAcc=0.8838
Epoch 9/30  TrainLoss=0.1851  ValLoss=0.1856  TrainAcc=0.9461  ValAcc=0.9461
Checkpoint saved! New best accuracy: 0.9461
Epoch 10/30  TrainLoss=0.1478  ValLoss=0.1333  TrainAcc=0.9601  V